# Step 4: RQ2 Full-List Summary

This notebook summarizes the RQ2 full-list condition without modifying any existing RQ2 notebooks or result files. It reads the existing `Top_24` batch outputs, compares each LLM-returned ranking with the model-grounded reference ranking, and reports Overlap@K and Kendall's tau at K = 5, 10, 15, 20, and 24.

Outputs are written only to a new folder: `Result/LLM/RQ2_FullList_Summary/`.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from scipy.stats import kendalltau

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending")


def find_local_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Data" / "ModifiedData").exists() and (candidate / "Result").exists():
            return candidate
    return start.parent if start.name == "Code" else start


BASE_DIR = COLAB_PROJECT_ROOT if COLAB_PROJECT_ROOT.exists() else find_local_project_root(PROJECT_ROOT)
RESULT_ROOT = BASE_DIR / "Result"
RESULT_PROJECT_DIR = RESULT_ROOT / "LendingClub" if (RESULT_ROOT / "LendingClub").exists() else RESULT_ROOT
SHAP_DIR = RESULT_PROJECT_DIR / "SHAP"
SHAP_RETRAINED_DIR = RESULT_PROJECT_DIR / "SHAP_retrained"
LLM_DIR = RESULT_PROJECT_DIR / "LLM"
SUMMARY_DIR = LLM_DIR / "RQ2_FullList_Summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = [5, 10, 15, 20, 24]
FULL_K = 24

print(f"Base directory: {BASE_DIR}")
print(f"LLM directory: {LLM_DIR}")
print(f"Summary output directory: {SUMMARY_DIR}")


## 2. Feature Name Mapping

In [ ]:
FEATURE_TO_NAME_MAP = {
    "home_ownership": "Home Ownership Status",
    "verification_status": "Income Verification Status",
    "purpose": "Loan Purpose",
    "area": "Borrower Area",
    "addr_state": "Borrower State",
    "term": "Number of Payments",
    "grade": "Loan Grade",
    "sub_grade": "Loan Sub Grade",
    "emp_length": "Employment Length",
    "pub_rec_bankruptcies": "Number of Public Bankruptcies",
    "funded_amnt": "Loan Amount",
    "int_rate": "Interest Rate",
    "installment": "Monthly Payment",
    "annual_inc": "Annual Income",
    "dti": "Debt to Income Ratio",
    "delinq_2yrs": "Number of Delinquencies in Past 2 Years",
    "fico_range_low": "Lowest FICO Score",
    "fico_range_high": "Highest FICO Score",
    "inq_last_6mths": "Number of Inquiries in Last 6 Months",
    "open_acc": "Number of Open Accounts",
    "pub_rec": "Number of Derogatory Public Records",
    "revol_bal": "Revolving Balance",
    "revol_util": "Revolving Utilization Rate",
    "total_acc": "Total Number of Accounts",
}
NAME_TO_FEATURE_MAP = {v: k for k, v in FEATURE_TO_NAME_MAP.items()}


## 3. RQ2 Full-List Result Folders

Both zero-shot and few-shot full-list folders are included. They are separated by the `Experiment` column.

In [ ]:
RUN_SPECS = [
    {
        "Experiment": "ZeroShot",
        "BaseModel": "Logistic Regression",
        "Top24Dir": LLM_DIR / "RQ2_Logistic_Redesigned" / "Top_24",
        "SamplePath": LLM_DIR / "500_samples_logistic.xlsx",
        "ReferencePath": SHAP_DIR / "Logistic_Contrib.xlsx",
        "ReferenceType": "wide_abs",
    },
    {
        "Experiment": "ZeroShot",
        "BaseModel": "XGBoost",
        "Top24Dir": LLM_DIR / "RQ2_XGBoost_Redesigned" / "Top_24",
        "SamplePath": LLM_DIR / "500_samples_xgboost_matched.xlsx",
        "ReferencePath": SHAP_RETRAINED_DIR / "XGBoost_SHAP_Aggregated.xlsx",
        "ReferenceType": "long_abs",
    },
    {
        "Experiment": "FewShot",
        "BaseModel": "Logistic Regression",
        "Top24Dir": LLM_DIR / "RQ2_Logistic_FewShot_Redesigned" / "Top_24",
        "SamplePath": LLM_DIR / "500_samples_logistic.xlsx",
        "ReferencePath": SHAP_DIR / "Logistic_Contrib.xlsx",
        "ReferenceType": "wide_abs",
    },
    {
        "Experiment": "FewShot",
        "BaseModel": "XGBoost",
        "Top24Dir": LLM_DIR / "RQ2_XGBoost_FewShot_Redesigned" / "Top_24",
        "SamplePath": LLM_DIR / "500_samples_xgboost_matched.xlsx",
        "ReferencePath": SHAP_RETRAINED_DIR / "XGBoost_SHAP_Aggregated.xlsx",
        "ReferenceType": "long_abs",
    },
]

for spec in RUN_SPECS:
    status = "FOUND" if spec["Top24Dir"].exists() else "MISSING"
    print(f"{status}: {spec['Experiment']} | {spec['BaseModel']} | {spec['Top24Dir']}")


## 4. Helper Functions

In [ ]:
def normalize_feature_text(text):
    text = str(text).strip()
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()


def feature_aliases(feature):
    aliases = {str(feature)}
    if feature in FEATURE_TO_NAME_MAP:
        aliases.add(FEATURE_TO_NAME_MAP[feature])
    if feature in NAME_TO_FEATURE_MAP:
        aliases.add(NAME_TO_FEATURE_MAP[feature])
    return {alias for alias in aliases if alias is not None and str(alias).strip()}


def build_feature_lookup(allowed_features):
    lookup = {}
    for feature in allowed_features:
        for alias in feature_aliases(feature):
            lookup[normalize_feature_text(alias)] = feature
    return lookup


def canonicalize_feature(feature, allowed_features):
    lookup = build_feature_lookup(allowed_features)
    normalized = normalize_feature_text(feature)
    if normalized in lookup:
        return lookup[normalized]
    for alias_key, raw_feature in lookup.items():
        if alias_key and alias_key in normalized:
            return raw_feature
    return None


def load_sample_df(sample_path):
    sample_df = pd.read_excel(sample_path)
    sample_df["Rowindex"] = sample_df["Rowindex"].astype(int)
    return sample_df


def load_reference(spec):
    if spec["ReferenceType"] == "wide_abs":
        ref = pd.read_excel(spec["ReferencePath"], index_col=0)
        ref.index = ref.index.astype(int)
        return ref

    ref = pd.read_excel(spec["ReferencePath"])
    ref["Rowindex"] = ref["Rowindex"].astype(int)
    if "AbsSHAPValue" not in ref.columns:
        if "AbsContribution" in ref.columns:
            ref["AbsSHAPValue"] = ref["AbsContribution"].abs()
        elif "SignedContribution" in ref.columns:
            ref["AbsSHAPValue"] = ref["SignedContribution"].abs()
        else:
            raise KeyError("Long reference table needs AbsSHAPValue, AbsContribution, or SignedContribution.")
    return ref


def reference_rank(rowindex, reference_df, reference_type):
    rowindex = int(rowindex)
    if reference_type == "wide_abs":
        row = reference_df.loc[rowindex]
        return row.abs().sort_values(ascending=False).index.astype(str).tolist()

    row = reference_df[reference_df["Rowindex"] == rowindex].copy()
    if row.empty:
        raise KeyError(f"Rowindex {rowindex} is missing from reference table.")
    return row.sort_values("AbsSHAPValue", ascending=False)["Feature"].astype(str).tolist()


def sorted_batch_files(model_dir):
    def batch_num(path):
        match = re.search(r"batch_(\d+)", path.stem)
        return int(match.group(1)) if match else 10**9
    return sorted(model_dir.glob("batch_*.xlsx"), key=batch_num)


def load_llm_batches(model_dir):
    frames = []
    for path in sorted_batch_files(model_dir):
        frame = pd.read_excel(path)
        frame["SourceFile"] = path.name
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["Rowindex", "FeatureRank", "Feature", "ReadableFeature", "SourceFile"])
    df = pd.concat(frames, ignore_index=True)
    df["Rowindex"] = df["Rowindex"].astype(int)
    df["FeatureRank"] = pd.to_numeric(df["FeatureRank"], errors="coerce")
    return df.sort_values(["Rowindex", "FeatureRank", "SourceFile"]).reset_index(drop=True)


def load_error_count(model_dir):
    error_path = model_dir / "errors.jsonl"
    if not error_path.exists():
        return 0
    with open(error_path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def overlap_topk(llm_ranking, ref_ranking, k):
    llm_topk = set(llm_ranking[:k])
    ref_topk = set(ref_ranking[:k])
    if not llm_topk:
        return np.nan
    return len(llm_topk & ref_topk) / k


def kendall_tau_topk(llm_ranking, ref_ranking, k):
    llm_topk = llm_ranking[:k]
    ref_rank_map = {feature: rank for rank, feature in enumerate(ref_ranking, start=1)}
    common_features = [feature for feature in llm_topk if feature in ref_rank_map]
    if len(common_features) < 2:
        return np.nan
    llm_ranks = list(range(1, len(common_features) + 1))
    ref_ranks = [ref_rank_map[feature] for feature in common_features]
    tau, _ = kendalltau(llm_ranks, ref_ranks)
    return tau


## 5. Compute Full-List Metrics by Sample

In [ ]:
def evaluate_full_list_for_spec(spec):
    if not spec["Top24Dir"].exists():
        print(f"Skipping missing Top_24 directory: {spec['Top24Dir']}")
        return pd.DataFrame()

    sample_df = load_sample_df(spec["SamplePath"])
    reference_df = load_reference(spec)
    rows = []

    for model_dir in sorted([p for p in spec["Top24Dir"].iterdir() if p.is_dir()]):
        llm_df = load_llm_batches(model_dir)
        error_count = load_error_count(model_dir)

        for rowindex in sample_df["Rowindex"].astype(int):
            ref_rank = reference_rank(rowindex, reference_df, spec["ReferenceType"])
            allowed_features = ref_rank[:FULL_K]
            model_rows = llm_df[llm_df["Rowindex"] == rowindex].sort_values("FeatureRank")

            raw_features = model_rows["Feature"].astype(str).tolist()
            canonical_features = []
            unknown_features = []
            for feature in raw_features:
                mapped = canonicalize_feature(feature, allowed_features)
                if mapped is None:
                    unknown_features.append(feature)
                else:
                    canonical_features.append(mapped)

            llm_rank = canonical_features[:FULL_K]
            duplicate_count = len(llm_rank) - len(set(llm_rank))
            parsed = len(llm_rank) > 0
            complete_full_list = len(llm_rank) == FULL_K and duplicate_count == 0 and len(unknown_features) == 0

            row = {
                "Experiment": spec["Experiment"],
                "BaseModel": spec["BaseModel"],
                "LLMResultDir": model_dir.name,
                "Rowindex": rowindex,
                "Parsed": parsed,
                "ReturnedCount": len(llm_rank),
                "CompleteFullList": complete_full_list,
                "UnknownFeatureCount": len(unknown_features),
                "DuplicateFeatureCount": duplicate_count,
                "UnknownFeatures": "; ".join(unknown_features),
                "ModelErrorLogCount": error_count,
            }

            for k in K_VALUES:
                row[f"overlap_top{k}"] = overlap_topk(llm_rank, ref_rank, k=k) if parsed else np.nan
                row[f"kendall_top{k}"] = kendall_tau_topk(llm_rank, ref_rank, k=k) if parsed else np.nan

            rows.append(row)

    return pd.DataFrame(rows)


all_detail_frames = []
for spec in RUN_SPECS:
    detail = evaluate_full_list_for_spec(spec)
    if not detail.empty:
        all_detail_frames.append(detail)

if all_detail_frames:
    full_detail_df = pd.concat(all_detail_frames, ignore_index=True)
else:
    full_detail_df = pd.DataFrame()

print(f"Sample-level rows: {len(full_detail_df)}")
display(full_detail_df.head())


## 6. Summary Table

In [ ]:
def summarize_full_list(detail_df):
    if detail_df.empty:
        return pd.DataFrame()

    metric_cols = [f"overlap_top{k}" for k in K_VALUES] + [f"kendall_top{k}" for k in K_VALUES]
    rows = []
    group_cols = ["Experiment", "BaseModel", "LLMResultDir"]
    for group_id, group in detail_df.groupby(group_cols, dropna=False):
        experiment, base_model, llm_dir = group_id
        row = {
            "Experiment": experiment,
            "BaseModel": base_model,
            "LLMResultDir": llm_dir,
            "N_Total": len(group),
            "N_Parsed": int(group["Parsed"].sum()),
            "ParsedRate": group["Parsed"].mean(),
            "CompleteFullListRate": group["CompleteFullList"].mean(),
            "MeanReturnedCount": group["ReturnedCount"].mean(),
            "TotalUnknownFeatureCount": int(group["UnknownFeatureCount"].sum()),
            "TotalDuplicateFeatureCount": int(group["DuplicateFeatureCount"].sum()),
            "TotalModelErrorLogCount": int(group["ModelErrorLogCount"].max()) if len(group) else 0,
        }
        for metric in metric_cols:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            row[f"Mean_{metric}"] = values.mean() if len(values) else np.nan
            row[f"Median_{metric}"] = values.median() if len(values) else np.nan
            row[f"Min_{metric}"] = values.min() if len(values) else np.nan
            row[f"Max_{metric}"] = values.max() if len(values) else np.nan
        rows.append(row)

    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


full_summary_df = summarize_full_list(full_detail_df)
display(full_summary_df)


## 7. Bootstrap 95% CI for Mean Metrics

In [ ]:
def bootstrap_mean_ci(values, n_bootstrap=5000, ci=0.95, random_state=42):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan, 0

    rng = np.random.default_rng(random_state)
    alpha = 1 - ci
    boot_idx = rng.integers(0, n, size=(n_bootstrap, n))
    boot_means = values[boot_idx].mean(axis=1)
    lower, upper = np.quantile(boot_means, [alpha / 2, 1 - alpha / 2])
    return float(values.mean()), float(lower), float(upper), n


def build_bootstrap_ci(detail_df, n_bootstrap=5000, ci=0.95, random_state=42):
    if detail_df.empty:
        return pd.DataFrame()

    metric_cols = [f"overlap_top{k}" for k in K_VALUES] + [f"kendall_top{k}" for k in K_VALUES]
    rows = []
    group_cols = ["Experiment", "BaseModel", "LLMResultDir"]
    for group_id, group in detail_df.groupby(group_cols, dropna=False):
        experiment, base_model, llm_dir = group_id
        for metric in metric_cols:
            mean, lower, upper, n = bootstrap_mean_ci(
                group[metric],
                n_bootstrap=n_bootstrap,
                ci=ci,
                random_state=random_state,
            )
            rows.append({
                "Experiment": experiment,
                "BaseModel": base_model,
                "LLMResultDir": llm_dir,
                "Metric": metric,
                "Mean": mean,
                "CI_Lower": lower,
                "CI_Upper": upper,
                "N": n,
                "BootstrapSamples": n_bootstrap,
                "CI_Level": ci,
            })

    return pd.DataFrame(rows).sort_values(group_cols + ["Metric"]).reset_index(drop=True)


full_bootstrap_ci_df = build_bootstrap_ci(full_detail_df, n_bootstrap=5000, ci=0.95, random_state=42)
display(full_bootstrap_ci_df)


## 8. Save New Summary Outputs

These files are new summary artifacts only. They do not overwrite any existing RQ2 results.

In [ ]:
detail_path = SUMMARY_DIR / "rq2_full_list_metrics_by_sample.xlsx"
summary_path = SUMMARY_DIR / "rq2_full_list_metrics_summary.xlsx"
ci_path = SUMMARY_DIR / "rq2_full_list_metrics_bootstrap_ci.xlsx"

if not full_detail_df.empty:
    full_detail_df.to_excel(detail_path, index=False)
    full_summary_df.to_excel(summary_path, index=False)
    full_bootstrap_ci_df.to_excel(ci_path, index=False)

    print(f"Saved sample-level file: {detail_path}")
    print(f"Saved summary file: {summary_path}")
    print(f"Saved bootstrap CI file: {ci_path}")
else:
    print("No full-list detail rows to save.")


## 10. Schema-Level Fidelity Check

This section mirrors the RQ1 schema-level fidelity table for the RQ2 full-list condition. It checks whether each expected output contains only valid supplied features, whether duplicate features appear, whether all 24 ranked positions are returned, and whether malformed JSON/parser failures occurred.

In [ ]:
def read_error_log(model_dir):
    error_path = model_dir / "errors.jsonl"
    if not error_path.exists():
        return pd.DataFrame(columns=["Rowindex", "ErrorType", "Error", "RawResponse"])

    rows = []
    with open(error_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                rows.append({"ErrorType": "MALFORMED_ERROR_LOG", "RawLine": line})

    if not rows:
        return pd.DataFrame(columns=["Rowindex", "ErrorType", "Error", "RawResponse"])

    df = pd.DataFrame(rows)
    if "Rowindex" in df.columns:
        df["Rowindex"] = pd.to_numeric(df["Rowindex"], errors="coerce").astype("Int64")
    return df


def collect_model_error_logs(run_specs):
    frames = []
    for spec in run_specs:
        top_dir = spec["Top24Dir"]
        if not top_dir.exists():
            continue
        for model_dir in sorted([p for p in top_dir.iterdir() if p.is_dir()]):
            err = read_error_log(model_dir)
            if err.empty:
                continue
            err["Experiment"] = spec["Experiment"]
            err["BaseModel"] = spec["BaseModel"]
            err["LLMResultDir"] = model_dir.name
            frames.append(err)
    if not frames:
        return pd.DataFrame(columns=["Experiment", "BaseModel", "LLMResultDir", "Rowindex", "ErrorType", "Error", "RawResponse"])
    return pd.concat(frames, ignore_index=True)


def build_schema_fidelity_table(detail_df, error_log_df):
    if detail_df.empty:
        return pd.DataFrame()

    rows = []
    group_cols = ["Experiment", "BaseModel"]
    for (experiment, base_model), group in detail_df.groupby(group_cols, dropna=False):
        # Count at the expected-output level: one expected output per sample per LLM.
        total_outputs = len(group)

        invalid_feature_cases = int((group["UnknownFeatureCount"] > 0).sum())
        duplicate_feature_cases = int((group["DuplicateFeatureCount"] > 0).sum())

        # Missing ranked positions includes parsed outputs that returned fewer than 24
        # valid positions. Completely unparsed malformed JSON outputs are counted
        # separately under Malformed JSON.
        missing_ranked_cases = int((group["Parsed"] & (group["ReturnedCount"] < FULL_K)).sum())

        err_group = error_log_df[
            (error_log_df["Experiment"] == experiment)
            & (error_log_df["BaseModel"] == base_model)
        ] if not error_log_df.empty else pd.DataFrame()
        malformed_json_cases = int((err_group["ErrorType"] == "FORMAT_ERROR").sum()) if not err_group.empty else 0

        complete_full_list_cases = int(group["CompleteFullList"].sum())
        rows.append({
            "Experiment": experiment,
            "Base Model": base_model,
            "Total Outputs": total_outputs,
            "Complete Full List": complete_full_list_cases,
            "Complete Full List Rate": complete_full_list_cases / total_outputs if total_outputs else np.nan,
            "Invalid Feature Names": invalid_feature_cases,
            "Duplicate Features": duplicate_feature_cases,
            "Missing Ranked Positions": missing_ranked_cases,
            "Malformed JSON": malformed_json_cases,
        })

    return pd.DataFrame(rows).sort_values(["Experiment", "Base Model"]).reset_index(drop=True)


full_error_log_df = collect_model_error_logs(RUN_SPECS)
rq2_full_schema_fidelity_df = build_schema_fidelity_table(full_detail_df, full_error_log_df)

display(rq2_full_schema_fidelity_df)

schema_path = SUMMARY_DIR / "rq2_full_list_schema_fidelity_table.xlsx"
rq2_full_schema_fidelity_df.to_excel(schema_path, index=False)
print(f"Saved schema-level fidelity table: {schema_path}")

# Optional LLM-level diagnostic table, useful for appendix or debugging.
rq2_full_schema_fidelity_by_llm = (
    full_detail_df
    .groupby(["Experiment", "BaseModel", "LLMResultDir"], dropna=False)
    .agg(
        TotalOutputs=("Rowindex", "size"),
        ParsedOutputs=("Parsed", "sum"),
        InvalidFeatureNames=("UnknownFeatureCount", lambda s: int((s > 0).sum())),
        DuplicateFeatures=("DuplicateFeatureCount", lambda s: int((s > 0).sum())),
        CompleteFullList=("CompleteFullList", "sum"),
        CompleteFullListRate=("CompleteFullList", "mean"),
        MissingRankedPositions=("ReturnedCount", lambda s: int((s < FULL_K).sum())),
    )
    .reset_index()
)

if not full_error_log_df.empty:
    malformed_by_llm = (
        full_error_log_df[full_error_log_df["ErrorType"] == "FORMAT_ERROR"]
        .groupby(["Experiment", "BaseModel", "LLMResultDir"])
        .size()
        .rename("MalformedJSON")
        .reset_index()
    )
    rq2_full_schema_fidelity_by_llm = rq2_full_schema_fidelity_by_llm.merge(
        malformed_by_llm,
        on=["Experiment", "BaseModel", "LLMResultDir"],
        how="left",
    )
else:
    rq2_full_schema_fidelity_by_llm["MalformedJSON"] = 0

rq2_full_schema_fidelity_by_llm["MalformedJSON"] = rq2_full_schema_fidelity_by_llm["MalformedJSON"].fillna(0).astype(int)

llm_schema_path = SUMMARY_DIR / "rq2_full_list_schema_fidelity_by_llm.xlsx"
rq2_full_schema_fidelity_by_llm.to_excel(llm_schema_path, index=False)
print(f"Saved LLM-level schema diagnostic table: {llm_schema_path}")
display(rq2_full_schema_fidelity_by_llm)


## 9. Manuscript-Ready Tables

For the full-list condition, Kendall's tau at 24 is the main full-ranking metric. Overlap@5 and Overlap@10 are useful for showing whether the highest-ranked features are recovered, while Overlap@24 is mostly a schema/membership check when all 24 valid features are returned.

In [ ]:
if full_bootstrap_ci_df.empty:
    report_table = pd.DataFrame()
else:
    report_metrics = ["overlap_top5", "overlap_top10", "kendall_top10", "kendall_top24"]
    report_table = full_bootstrap_ci_df[full_bootstrap_ci_df["Metric"].isin(report_metrics)].copy()
    report_table["Mean (95% CI)"] = report_table.apply(
        lambda r: f"{r['Mean']:.3f} ({r['CI_Lower']:.3f}, {r['CI_Upper']:.3f})" if pd.notna(r["Mean"]) else "NA",
        axis=1,
    )
    report_table = report_table[["Experiment", "BaseModel", "LLMResultDir", "Metric", "N", "Mean (95% CI)"]]
    display(report_table)

    report_path = SUMMARY_DIR / "rq2_full_list_report_table_long.xlsx"
    report_table.to_excel(report_path, index=False)
    print(f"Saved manuscript-ready long table: {report_path}")

    # Do not include metric-specific N in the pivot index. Kendall metrics can have
    # fewer non-missing rows than overlap metrics because Kendall requires at least
    # two common features, so using N in the index would split one model into rows.
    wide_report = report_table.pivot_table(
        index=["Experiment", "BaseModel", "LLMResultDir"],
        columns="Metric",
        values="Mean (95% CI)",
        aggfunc="first",
    ).reset_index()

    n_summary = (
        full_detail_df
        .groupby(["Experiment", "BaseModel", "LLMResultDir"], dropna=False)
        .agg(
            N_Total=("Rowindex", "size"),
            N_Parsed=("Parsed", "sum"),
            CompleteFullListRate=("CompleteFullList", "mean"),
        )
        .reset_index()
    )
    n_summary["N_Parsed"] = n_summary["N_Parsed"].astype(int)
    n_summary["CompleteFullListRate"] = n_summary["CompleteFullListRate"].round(4)

    wide_report = n_summary.merge(
        wide_report,
        on=["Experiment", "BaseModel", "LLMResultDir"],
        how="left",
    )
    display(wide_report)

    wide_report_path = SUMMARY_DIR / "rq2_full_list_report_table_wide.xlsx"
    wide_report.to_excel(wide_report_path, index=False)
    print(f"Saved manuscript-ready wide table: {wide_report_path}")


In [ ]:
report_table[report_table.LLMResultDir.str.contains("gpt-4", case=False, na=False)]

## 10. fidelty Check

In [ ]:
def read_error_log(model_dir):
    error_path = model_dir / "errors.jsonl"
    if not error_path.exists():
        return pd.DataFrame(columns=["Rowindex", "ErrorType", "Error", "RawResponse"])

    rows = []
    with open(error_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                rows.append({"ErrorType": "MALFORMED_ERROR_LOG", "RawLine": line})

    if not rows:
        return pd.DataFrame(columns=["Rowindex", "ErrorType", "Error", "RawResponse"])

    df = pd.DataFrame(rows)
    if "Rowindex" in df.columns:
        df["Rowindex"] = pd.to_numeric(df["Rowindex"], errors="coerce").astype("Int64")
    return df


def collect_model_error_logs(run_specs):
    frames = []
    for spec in run_specs:
        top_dir = spec["Top24Dir"]
        if not top_dir.exists():
            continue

        for model_dir in sorted([p for p in top_dir.iterdir() if p.is_dir()]):
            err = read_error_log(model_dir)
            if err.empty:
                continue

            err["Experiment"] = spec["Experiment"]
            err["BaseModel"] = spec["BaseModel"]
            err["LLMResultDir"] = model_dir.name
            frames.append(err)

    if not frames:
        return pd.DataFrame(
            columns=[
                "Experiment", "BaseModel", "LLMResultDir",
                "Rowindex", "ErrorType", "Error", "RawResponse"
            ]
        )

    return pd.concat(frames, ignore_index=True)


def build_schema_fidelity_table(detail_df, error_log_df):
    if detail_df.empty:
        return pd.DataFrame()

    rows = []
    group_cols = ["Experiment", "BaseModel"]

    for (experiment, base_model), group in detail_df.groupby(group_cols, dropna=False):
        total_outputs = len(group)

        complete_full_list_cases = int(group["CompleteFullList"].sum())
        invalid_feature_cases = int((group["UnknownFeatureCount"] > 0).sum())
        duplicate_feature_cases = int((group["DuplicateFeatureCount"] > 0).sum())

        # Parsed outputs that returned fewer than 24 valid ranked positions.
        # Completely unparsed outputs are counted under Malformed JSON.
        missing_ranked_cases = int(
            (group["Parsed"] & (group["ReturnedCount"] < FULL_K)).sum()
        )

        if not error_log_df.empty:
            err_group = error_log_df[
                (error_log_df["Experiment"] == experiment)
                & (error_log_df["BaseModel"] == base_model)
            ]
            malformed_json_cases = int((err_group["ErrorType"] == "FORMAT_ERROR").sum())
        else:
            malformed_json_cases = 0

        rows.append({
            "Experiment": experiment,
            "Base Model": base_model,
            "Total Outputs": total_outputs,
            "Complete Full List": complete_full_list_cases,
            "Complete Full List Rate": complete_full_list_cases / total_outputs if total_outputs else np.nan,
            "Invalid Feature Names": invalid_feature_cases,
            "Duplicate Features": duplicate_feature_cases,
            "Missing Ranked Positions": missing_ranked_cases,
            "Malformed JSON": malformed_json_cases,
        })

    return (
        pd.DataFrame(rows)
        .sort_values(["Experiment", "Base Model"])
        .reset_index(drop=True)
    )


def build_schema_fidelity_by_llm(detail_df, error_log_df):
    if detail_df.empty:
        return pd.DataFrame()

    out = (
        detail_df
        .groupby(["Experiment", "BaseModel", "LLMResultDir"], dropna=False)
        .agg(
            TotalOutputs=("Rowindex", "size"),
            ParsedOutputs=("Parsed", "sum"),
            CompleteFullList=("CompleteFullList", "sum"),
            CompleteFullListRate=("CompleteFullList", "mean"),
            InvalidFeatureNames=("UnknownFeatureCount", lambda s: int((s > 0).sum())),
            DuplicateFeatures=("DuplicateFeatureCount", lambda s: int((s > 0).sum())),
            MissingRankedPositions=("ReturnedCount", lambda s: int((s < FULL_K).sum())),
        )
        .reset_index()
    )

    out["ParsedOutputs"] = out["ParsedOutputs"].astype(int)
    out["CompleteFullList"] = out["CompleteFullList"].astype(int)
    out["CompleteFullListRate"] = out["CompleteFullListRate"].round(4)

    if not error_log_df.empty:
        malformed_by_llm = (
            error_log_df[error_log_df["ErrorType"] == "FORMAT_ERROR"]
            .groupby(["Experiment", "BaseModel", "LLMResultDir"])
            .size()
            .rename("MalformedJSON")
            .reset_index()
        )

        out = out.merge(
            malformed_by_llm,
            on=["Experiment", "BaseModel", "LLMResultDir"],
            how="left",
        )
    else:
        out["MalformedJSON"] = 0

    out["MalformedJSON"] = out["MalformedJSON"].fillna(0).astype(int)

    return out.sort_values(["Experiment", "BaseModel", "LLMResultDir"]).reset_index(drop=True)


full_error_log_df = collect_model_error_logs(RUN_SPECS)

rq2_full_schema_fidelity_df = build_schema_fidelity_table(
    full_detail_df,
    full_error_log_df,
)

rq2_full_schema_fidelity_by_llm = build_schema_fidelity_by_llm(
    full_detail_df,
    full_error_log_df,
)

display(rq2_full_schema_fidelity_df)
display(rq2_full_schema_fidelity_by_llm)

schema_path = SUMMARY_DIR / "rq2_full_list_schema_fidelity_table.xlsx"
llm_schema_path = SUMMARY_DIR / "rq2_full_list_schema_fidelity_by_llm.xlsx"

rq2_full_schema_fidelity_df.to_excel(schema_path, index=False)
rq2_full_schema_fidelity_by_llm.to_excel(llm_schema_path, index=False)

print(f"Saved schema-level fidelity table: {schema_path}")
print(f"Saved LLM-level schema diagnostic table: {llm_schema_path}")

In [ ]:
rq2_full_schema_fidelity_by_llm[rq2_full_schema_fidelity_by_llm.Experiment == "FewShot"]


## Gemini Returned Feature Count Diagnostic

This diagnostic checks how many features Gemini returned in the RQ2 full-list condition. It helps distinguish poor ranking alignment from failure to return a complete 24-feature list. Run this after `full_detail_df` has been created.


In [ ]:
# Gemini returned feature-count diagnostic for RQ2 full-list condition.
# Run this after the cell that creates full_detail_df.

gemini_full_detail_df = full_detail_df[
    full_detail_df["LLMResultDir"].str.contains("gemini", case=False, na=False)
].copy()

if gemini_full_detail_df.empty:
    print("No Gemini rows found in full_detail_df. Run the full-list metric construction cells first.")
else:
    gemini_length_summary_df = (
        gemini_full_detail_df
        .groupby(["Experiment", "BaseModel", "LLMResultDir"], dropna=False)
        .agg(
            TotalOutputs=("Rowindex", "size"),
            ParsedOutputs=("Parsed", "sum"),
            MeanReturnedCount=("ReturnedCount", "mean"),
            MedianReturnedCount=("ReturnedCount", "median"),
            MinReturnedCount=("ReturnedCount", "min"),
            MaxReturnedCount=("ReturnedCount", "max"),
            CompleteFullList=("CompleteFullList", "sum"),
            CompleteFullListRate=("CompleteFullList", "mean"),
        )
        .reset_index()
    )

    gemini_length_summary_df["ParsedOutputs"] = gemini_length_summary_df["ParsedOutputs"].astype(int)
    gemini_length_summary_df["CompleteFullList"] = gemini_length_summary_df["CompleteFullList"].astype(int)
    for col in ["MeanReturnedCount", "MedianReturnedCount", "CompleteFullListRate"]:
        gemini_length_summary_df[col] = gemini_length_summary_df[col].round(4)

    display(gemini_length_summary_df)

    gemini_returned_count_distribution_df = (
        gemini_full_detail_df
        .groupby(["Experiment", "BaseModel", "LLMResultDir", "ReturnedCount"], dropna=False)
        .size()
        .rename("Count")
        .reset_index()
        .sort_values(["Experiment", "BaseModel", "LLMResultDir", "ReturnedCount"])
    )
    gemini_returned_count_distribution_df["Rate"] = (
        gemini_returned_count_distribution_df
        .groupby(["Experiment", "BaseModel", "LLMResultDir"])["Count"]
        .transform(lambda s: s / s.sum())
    ).round(4)

    display(gemini_returned_count_distribution_df)

    gemini_returned_count_wide_df = gemini_returned_count_distribution_df.pivot_table(
        index=["Experiment", "BaseModel", "LLMResultDir"],
        columns="ReturnedCount",
        values="Count",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()

    display(gemini_returned_count_wide_df)

    gemini_length_summary_path = SUMMARY_DIR / "rq2_full_list_gemini_returned_count_summary.xlsx"
    gemini_distribution_path = SUMMARY_DIR / "rq2_full_list_gemini_returned_count_distribution.xlsx"
    gemini_wide_path = SUMMARY_DIR / "rq2_full_list_gemini_returned_count_distribution_wide.xlsx"

    gemini_length_summary_df.to_excel(gemini_length_summary_path, index=False)
    gemini_returned_count_distribution_df.to_excel(gemini_distribution_path, index=False)
    gemini_returned_count_wide_df.to_excel(gemini_wide_path, index=False)

    print(f"Saved Gemini returned-count summary: {gemini_length_summary_path}")
    print(f"Saved Gemini returned-count distribution: {gemini_distribution_path}")
    print(f"Saved Gemini returned-count wide distribution: {gemini_wide_path}")


## ChatGPT and Claude Overlap@K Summary

This block summarizes only ChatGPT and Claude for `Overlap@5`, `Overlap@10`, `Overlap@15`, and `Overlap@20`. It uses the already-computed bootstrap CI table and does not modify the original RQ2 result files.



In [ ]:
# ChatGPT and Claude Overlap@K summary for RQ2 full-list condition.
# This block reuses full_bootstrap_ci_df and full_detail_df generated above.

SELECTED_OVERLAP_K = [5, 10, 15, 20]
SELECTED_OVERLAP_METRICS = [f"overlap_top{k}" for k in SELECTED_OVERLAP_K]


def label_chatgpt_or_claude(llm_result_dir):
    name = str(llm_result_dir).lower()
    if "gpt" in name:
        return "ChatGPT"
    if "claude" in name or "anthropic" in name:
        return "Claude"
    return None


if full_bootstrap_ci_df.empty:
    chatgpt_claude_overlap_long_df = pd.DataFrame()
    chatgpt_claude_overlap_wide_df = pd.DataFrame()
    print("full_bootstrap_ci_df is empty. Run the metric and bootstrap sections first.")
else:
    chatgpt_claude_overlap_long_df = full_bootstrap_ci_df[
        full_bootstrap_ci_df["Metric"].isin(SELECTED_OVERLAP_METRICS)
    ].copy()

    chatgpt_claude_overlap_long_df["LLM"] = chatgpt_claude_overlap_long_df["LLMResultDir"].apply(label_chatgpt_or_claude)
    chatgpt_claude_overlap_long_df = chatgpt_claude_overlap_long_df[
        chatgpt_claude_overlap_long_df["LLM"].notna()
    ].copy()

    chatgpt_claude_overlap_long_df["K"] = (
        chatgpt_claude_overlap_long_df["Metric"]
        .str.extract(r"overlap_top(\d+)")[0]
        .astype(int)
    )
    chatgpt_claude_overlap_long_df["MetricLabel"] = "Overlap@" + chatgpt_claude_overlap_long_df["K"].astype(str)
    chatgpt_claude_overlap_long_df["Mean (95% CI)"] = chatgpt_claude_overlap_long_df.apply(
        lambda r: f"{r['Mean']:.3f} ({r['CI_Lower']:.3f}, {r['CI_Upper']:.3f})" if pd.notna(r["Mean"]) else "NA",
        axis=1,
    )

    chatgpt_claude_overlap_long_df = chatgpt_claude_overlap_long_df[
        [
            "Experiment",
            "BaseModel",
            "LLM",
            "LLMResultDir",
            "MetricLabel",
            "N",
            "Mean",
            "CI_Lower",
            "CI_Upper",
            "Mean (95% CI)",
        ]
    ].sort_values(["Experiment", "BaseModel", "LLM", "MetricLabel"]).reset_index(drop=True)

    display(chatgpt_claude_overlap_long_df)

    chatgpt_claude_overlap_wide_df = chatgpt_claude_overlap_long_df.pivot_table(
        index=["Experiment", "BaseModel", "LLM", "LLMResultDir"],
        columns="MetricLabel",
        values="Mean (95% CI)",
        aggfunc="first",
    ).reset_index()

    # Keep columns in K order instead of alphabetical order.
    ordered_metric_cols = [f"Overlap@{k}" for k in SELECTED_OVERLAP_K]
    chatgpt_claude_overlap_wide_df = chatgpt_claude_overlap_wide_df[
        ["Experiment", "BaseModel", "LLM", "LLMResultDir"] + ordered_metric_cols
    ]

    display(chatgpt_claude_overlap_wide_df)

    long_path = SUMMARY_DIR / "rq2_full_list_chatgpt_claude_overlap_at_k_long.xlsx"
    wide_path = SUMMARY_DIR / "rq2_full_list_chatgpt_claude_overlap_at_k_wide.xlsx"
    chatgpt_claude_overlap_long_df.to_excel(long_path, index=False)
    chatgpt_claude_overlap_wide_df.to_excel(wide_path, index=False)
    print(f"Saved ChatGPT/Claude Overlap@K long table: {long_path}")
    print(f"Saved ChatGPT/Claude Overlap@K wide table: {wide_path}")



In [ ]:
chatgpt_claude_overlap_wide_df[chatgpt_claude_overlap_wide_df.BaseModel != "XGBoost"]

## ChatGPT and Claude Kendall Tau@K Summary

This block summarizes only ChatGPT and Claude for Kendall's tau at `K = 5, 10, 15, 20`. It uses the already-computed bootstrap CI table and does not modify the original RQ2 result files.



In [ ]:
# ChatGPT and Claude Kendall's tau@K summary for RQ2 full-list condition.
# This block reuses full_bootstrap_ci_df generated above.

SELECTED_KENDALL_K = [5, 10, 15, 20]
SELECTED_KENDALL_METRICS = [f"kendall_top{k}" for k in SELECTED_KENDALL_K]


def label_chatgpt_or_claude(llm_result_dir):
    name = str(llm_result_dir).lower()
    if "gpt" in name:
        return "ChatGPT"
    if "claude" in name or "anthropic" in name:
        return "Claude"
    return None


if full_bootstrap_ci_df.empty:
    chatgpt_claude_kendall_long_df = pd.DataFrame()
    chatgpt_claude_kendall_wide_df = pd.DataFrame()
    print("full_bootstrap_ci_df is empty. Run the metric and bootstrap sections first.")
else:
    chatgpt_claude_kendall_long_df = full_bootstrap_ci_df[
        full_bootstrap_ci_df["Metric"].isin(SELECTED_KENDALL_METRICS)
    ].copy()

    chatgpt_claude_kendall_long_df["LLM"] = chatgpt_claude_kendall_long_df["LLMResultDir"].apply(label_chatgpt_or_claude)
    chatgpt_claude_kendall_long_df = chatgpt_claude_kendall_long_df[
        chatgpt_claude_kendall_long_df["LLM"].notna()
    ].copy()

    chatgpt_claude_kendall_long_df["K"] = (
        chatgpt_claude_kendall_long_df["Metric"]
        .str.extract(r"kendall_top(\d+)")[0]
        .astype(int)
    )
    chatgpt_claude_kendall_long_df["MetricLabel"] = "KendallTau@" + chatgpt_claude_kendall_long_df["K"].astype(str)
    chatgpt_claude_kendall_long_df["Mean (95% CI)"] = chatgpt_claude_kendall_long_df.apply(
        lambda r: f"{r['Mean']:.3f} ({r['CI_Lower']:.3f}, {r['CI_Upper']:.3f})" if pd.notna(r["Mean"]) else "NA",
        axis=1,
    )

    chatgpt_claude_kendall_long_df = (
        chatgpt_claude_kendall_long_df
        .sort_values(["Experiment", "BaseModel", "LLM", "K"])
        [
            [
                "Experiment",
                "BaseModel",
                "LLM",
                "LLMResultDir",
                "MetricLabel",
                "N",
                "Mean",
                "CI_Lower",
                "CI_Upper",
                "Mean (95% CI)",
            ]
        ]
        .reset_index(drop=True)
    )

    display(chatgpt_claude_kendall_long_df)

    chatgpt_claude_kendall_wide_df = chatgpt_claude_kendall_long_df.pivot_table(
        index=["Experiment", "BaseModel", "LLM", "LLMResultDir"],
        columns="MetricLabel",
        values="Mean (95% CI)",
        aggfunc="first",
    ).reset_index()

    # Keep columns in K order instead of alphabetical order.
    ordered_metric_cols = [f"KendallTau@{k}" for k in SELECTED_KENDALL_K]
    chatgpt_claude_kendall_wide_df = chatgpt_claude_kendall_wide_df[
        ["Experiment", "BaseModel", "LLM", "LLMResultDir"] + ordered_metric_cols
    ]

    display(chatgpt_claude_kendall_wide_df)

    long_path = SUMMARY_DIR / "rq2_full_list_chatgpt_claude_kendall_tau_at_k_long.xlsx"
    wide_path = SUMMARY_DIR / "rq2_full_list_chatgpt_claude_kendall_tau_at_k_wide.xlsx"
    chatgpt_claude_kendall_long_df.to_excel(long_path, index=False)
    chatgpt_claude_kendall_wide_df.to_excel(wide_path, index=False)
    print(f"Saved ChatGPT/Claude Kendall tau@K long table: {long_path}")
    print(f"Saved ChatGPT/Claude Kendall tau@K wide table: {wide_path}")



In [ ]:
chatgpt_claude_kendall_wide_df[chatgpt_claude_kendall_wide_df.BaseModel == "XGBoost"]